In [26]:
import pandas as pd
import numpy as np
import itertools
import time
import statsmodels.api as sm                    # Для OLS-регрессии
from statsmodels.tsa.stattools import coint    # Для теста коинтеграции


In [27]:
#Default Settings
p_value = 0.05
min_correlation = 0.5
min_half_life = 1.0
max_half_life = 42.0
min_hurst = 0.0
max_hurst = 0.5
scenario_name = 'DEFAULT'

#settings.csv
defaults = {
    'p_value': p_value,
    'min_correlation': min_correlation,
    'min_half_life': min_half_life,
    'max_half_life': max_half_life,
    'min_hurst': min_hurst,
    'max_hurst': max_hurst,
}
try:
    settings_df = pd.read_csv('CSV/settings.csv', sep=';', index_col='Parameter')
    for key in defaults:
        if key in settings_df.index:
            defaults[key] = float(settings_df.loc[key, 'Value'])
            print(f'[S] {key}:', defaults[key])
        else:
            print(f'[D] {key}:', defaults[key])
    if 'scenario_name' in settings_df.index:
        scenario_name = str(settings_df.loc['scenario_name', 'Value'])
        print('[S] scenario_name:', scenario_name)
    else:
        print('[D] scenario_name:', scenario_name)
except Exception as e:
    print('[D] settings.csv не прочитан, дефолты:', defaults)

p_value = defaults['p_value']
min_correlation = defaults['min_correlation']
min_half_life = defaults['min_half_life']
max_half_life = defaults['max_half_life']
min_hurst = defaults['min_hurst']
max_hurst = defaults['max_hurst']

[S] p_value: 0.01
[S] min_correlation: 0.3
[S] min_half_life: 3.0
[S] max_half_life: 8.0
[S] min_hurst: 0.0
[S] max_hurst: 0.28
[S] scenario_name: ULTRA


In [28]:
#Loading Master DF
master_df = pd.read_excel('DATA/FULL_Trading_Calendar.xlsx')
master_df['Date'] = pd.to_datetime(master_df['Date'])
master_df.set_index('Date', inplace=True)

print('[OK]')
print('Days:',master_df.shape[0])
print('Avaliable Tickers:',master_df.shape[1])

[OK]
Days: 2765
Avaliable Tickers: 525


In [29]:
#загружает расписание периодов для тестирования торговой стратегии
#подготавливает даты для работы с временными рядами.

try:
    schedule_df = pd.read_csv('CSV/schedule.csv', sep =';')

    schedule_df['IS_Start'] = pd.to_datetime(schedule_df['IS_Start'], dayfirst=True)
    schedule_df['IS_End'] = pd.to_datetime(schedule_df['IS_End'], dayfirst=True)

    schedule_df['OOS_Start'] = pd.to_datetime(schedule_df['OOS_Start'], dayfirst=True)
    schedule_df['OOS_End'] = pd.to_datetime(schedule_df['OOS_End'], dayfirst=True)

    print('[OK]')
    print('Итераций в расписании:', len(schedule_df))
except Exception as e:
    print('[ERROR]:', e)

[OK]
Итераций в расписании: 108


In [30]:
# Восстанавливаем тип актива для каждого тикера (нужно для filter_correlation)
tickers_df = pd.read_csv('CSV/Tickers.csv', keep_default_na=False)
ticker_type_map = {}
for col in tickers_df.columns:
    for ticker in tickers_df[col]:
        clean_ticker = ticker.strip().replace('.', '-') if col == 'EQUITY' else ticker.strip()
        if clean_ticker:
            ticker_type_map[clean_ticker] = col

print('[OK] Тикеров с типом:', len(ticker_type_map))

[OK] Тикеров с типом: 525


In [31]:
def filter_correlation(is_data, ticker_type_map, min_corr=0.5):
    """
    Фильтр 1: корреляция на лог-доходностях.
    Исключает только equity-equity, остальное разрешено.

    Возвращает (DataFrame прошедших порог пар, общее число проверенных
    не-equity-equity пар) — второе число нужно для статистики воронки
    фильтров (сколько пар было ДО отсева по корреляции).
    """
    log_returns = np.log(is_data / is_data.shift(1)).dropna(how='all')
    valid_tickers = log_returns.dropna(axis=1, how='any').columns.tolist()

    results = []
    n_total_pairs = 0
    for t1, t2 in itertools.combinations(valid_tickers, 2):
        type1 = ticker_type_map.get(t1, 'UNKNOWN')
        type2 = ticker_type_map.get(t2, 'UNKNOWN')
        if type1 == 'EQUITY' and type2 == 'EQUITY':
            continue  # искл. только equity-equity
        n_total_pairs += 1

        corr = log_returns[t1].corr(log_returns[t2])
        if pd.notna(corr) and abs(corr) >= min_corr:
            results.append({
                'Asset_A': t1, 'Asset_B': t2,
                'Type_A': type1, 'Type_B': type2,
                'Pearson_Corr': round(corr, 4)
            })

    return pd.DataFrame(results), n_total_pairs


def filter_cointegration(candidates_df, is_data, p_value=0.05):
    """
    Фильтр 2: Энгл-Грейнджер, тестируем ОБА направления (A~B и B~A).

    coint() асимметричен: результат зависит от того, какой ряд считается
    зависимым, а какой — регрессором (Liu, гл. 8). Chan (стр. 71-72, пример
    EWA/EWC) прямо описывает эту же проблему для CADF-теста и рекомендует:
    'try each variable as independent and see which order gives the best
    (most negative) t-statistic, and use that order to obtain the hedge
    ratio'. Здесь делаем ровно это — считаем оба направления и берём
    лучшее (min p-value), без дополнительной коррекции порога (следуем
    Chan буквально, без отступлений).

    Победившее направление становится Asset_A (зависимая) / Asset_B
    (регрессор) — это важно, т.к. _spread() ниже по пайплайну строит
    OLS(Asset_A, Asset_B), и должен использовать то же направление,
    на котором коинтеграция была подтверждена.
    """
    if candidates_df.empty:
        return candidates_df

    rows = []
    for _, row in candidates_df.iterrows():
        t1, t2 = row['Asset_A'], row['Asset_B']
        try:
            s1, s2 = is_data[t1].dropna(), is_data[t2].dropna()
            idx = s1.index.intersection(s2.index)
            s1, s2 = s1.loc[idx], s2.loc[idx]
            _, p_ab, _ = coint(s1, s2)   # t1 зависимая, t2 регрессор
            _, p_ba, _ = coint(s2, s1)   # t2 зависимая, t1 регрессор
        except Exception:
            p_ab, p_ba = np.nan, np.nan

        if pd.isna(p_ab) and pd.isna(p_ba):
            p_best, swap = np.nan, False
        elif pd.isna(p_ba) or (not pd.isna(p_ab) and p_ab <= p_ba):
            p_best, swap = p_ab, False
        else:
            p_best, swap = p_ba, True

        new_row = row.copy()
        if swap:
            new_row['Asset_A'], new_row['Asset_B'] = row['Asset_B'], row['Asset_A']
            new_row['Type_A'], new_row['Type_B'] = row['Type_B'], row['Type_A']
        new_row['P_Value'] = p_best
        rows.append(new_row)

    result = pd.DataFrame(rows).reset_index(drop=True)
    result = result[result['P_Value'] <= p_value].reset_index(drop=True)
    return result


def _spread(is_data, t1, t2):
    """
    Строит спред пары через OLS: spread = S1 - beta*S2 - intercept.
    Возвращает (spread, beta, intercept) — beta/intercept обязательны для
    OOS-движка: hedge ratio фиксируется на IS-периоде и используется без
    переобучения на OOS-данных (Chan, стр. 65-66), поэтому их нужно
    сохранить в итоговый файл, а не просто использовать внутри фильтров.
    """
    s1, s2 = is_data[t1].dropna(), is_data[t2].dropna()
    idx = s1.index.intersection(s2.index)
    s1, s2 = s1.loc[idx], s2.loc[idx]
    model = sm.OLS(s1, sm.add_constant(s2)).fit()
    beta = model.params.iloc[1]
    intercept = model.params.iloc[0]
    spread = s1 - beta * s2 - intercept
    return spread, beta, intercept


def filter_half_life(candidates_df, is_data, min_hl=1.0, max_hl=42.0):
    """
    Фильтр 3: Half-Life — скорость возврата спреда к среднему (Chan, 2013).
    HL = ln(2) / |λ|, где λ из регрессии Δspread(t) = λ*spread(t-1) + ε.

    Здесь же сохраняем Beta/Intercept (hedge ratio) пары — нужны OOS-движку
    для восстановления того же спреда на OOS-данных (см. _spread()).
    """
    if candidates_df.empty:
        return candidates_df

    half_lives, betas, intercepts = [], [], []
    for _, row in candidates_df.iterrows():
        t1, t2 = row['Asset_A'], row['Asset_B']
        try:
            spread, beta, intercept = _spread(is_data, t1, t2)

            spread_lag = spread.shift(1).dropna()
            spread_diff = spread.diff().dropna()
            common = spread_lag.index.intersection(spread_diff.index)
            y = spread_diff.loc[common]
            x = sm.add_constant(spread_lag.loc[common])
            lambda_val = sm.OLS(y, x).fit().params.iloc[1]

            hl = -np.log(2) / lambda_val if lambda_val < 0 else 999.0
        except Exception:
            hl, beta, intercept = 999.0, np.nan, np.nan
        half_lives.append(round(float(hl), 2))
        betas.append(round(float(beta), 6) if pd.notna(beta) else np.nan)
        intercepts.append(round(float(intercept), 6) if pd.notna(intercept) else np.nan)

    result = candidates_df.copy()
    result['Half_Life'] = half_lives
    result['Beta'] = betas
    result['Intercept'] = intercepts
    result = result[(result['Half_Life'] >= min_hl) & (result['Half_Life'] <= max_hl)].reset_index(drop=True)
    return result


def filter_hurst(candidates_df, is_data, min_hurst=0.0, max_hurst=0.5):
    """
    Фильтр 4: Hurst Exponent — подтверждение mean reversion (H < 0.5).
    Обобщённая экспонента Хёрста (generalized Hurst exponent, Chan 2013),
    метод дисперсии лагов: std(S(t)-S(t-lag)) ~ lag^H. НЕ классический R/S-анализ.
    """
    if candidates_df.empty:
        return candidates_df

    hursts = []
    for _, row in candidates_df.iterrows():
        t1, t2 = row['Asset_A'], row['Asset_B']
        try:
            spread, _, _ = _spread(is_data, t1, t2)
            ts = spread.values
            n = len(ts)
            lags = [l for l in [2, 4, 8, 16, 32, 64, 128] if l < n // 2]
            if n < 20 or len(lags) < 2:
                h = 0.5
            else:
                tau = [np.std(np.subtract(ts[lag:], ts[:-lag])) for lag in lags]
                h = 0.5 if any(t == 0 for t in tau) else np.polyfit(np.log(lags), np.log(tau), 1)[0]
        except Exception:
            h = 0.5
        hursts.append(round(float(h), 4))

    result = candidates_df.copy()
    result['Hurst'] = hursts
    result = result[(result['Hurst'] >= min_hurst) & (result['Hurst'] <= max_hurst)].reset_index(drop=True)
    return result

In [32]:
# Цикл по всем IS-окнам из schedule.csv.
# На каждой итерации фильтры считаются заново, с нуля (без переиспользования).
# Параллельно собираем статистику воронки: сколько пар прошло через каждый фильтр.

all_results = []
funnel_stats = []
t0 = time.time()

for idx, row in schedule_df.iterrows():
    iteration_num = idx + 1
    is_start = row['IS_Start']
    is_end = row['IS_End']
    is_data = master_df.loc[is_start:is_end]

    if is_data.empty:
        print(f"Итерация {iteration_num}: нет данных, пропуск")
        continue

    candidates, n_total_pairs = filter_correlation(is_data, ticker_type_map, min_corr=min_correlation)
    if candidates.empty:
        print(f"Итерация {iteration_num} ({is_start.date()} -> {is_end.date()}): 0 пар после корреляции")
        funnel_stats.append({
            'Iteration': iteration_num,
            'N_Total_Pairs': n_total_pairs,
            'N_After_Correlation': 0,
            'N_After_Cointegration': 0,
            'N_After_HalfLife': 0,
            'N_After_Hurst': 0,
        })
        continue
    candidates.insert(0, 'Iteration', iteration_num)

    cointegrated = filter_cointegration(candidates, is_data, p_value=p_value)
    hl_filtered = filter_half_life(cointegrated, is_data, min_hl=min_half_life, max_hl=max_half_life)
    hurst_filtered = filter_hurst(hl_filtered, is_data, min_hurst=min_hurst, max_hurst=max_hurst)

    print(f"Итерация {iteration_num} ({is_start.date()} -> {is_end.date()}): "
          f"кандидатов={n_total_pairs} -> корр={len(candidates)} -> коинт={len(cointegrated)} -> "
          f"HL={len(hl_filtered)} -> Hurst={len(hurst_filtered)}")

    funnel_stats.append({
        'Iteration': iteration_num,
        'N_Total_Pairs': n_total_pairs,
        'N_After_Correlation': len(candidates),
        'N_After_Cointegration': len(cointegrated),
        'N_After_HalfLife': len(hl_filtered),
        'N_After_Hurst': len(hurst_filtered),
    })

    all_results.append(hurst_filtered)

elapsed = time.time() - t0
print(f"\n[OK] Цикл завершён: {len(schedule_df)} итераций за {elapsed/60:.1f} мин")

Итерация 1 (2016-01-01 -> 2016-12-31): кандидатов=7544 -> корр=153 -> коинт=0 -> HL=0 -> Hurst=0
Итерация 2 (2016-02-01 -> 2017-01-31): кандидатов=7544 -> корр=129 -> коинт=3 -> HL=2 -> Hurst=1
Итерация 3 (2016-03-01 -> 2017-02-28): кандидатов=7544 -> корр=89 -> коинт=1 -> HL=1 -> Hurst=0
Итерация 4 (2016-04-01 -> 2017-03-31): кандидатов=7544 -> корр=90 -> коинт=2 -> HL=1 -> Hurst=1
Итерация 5 (2016-05-01 -> 2017-04-30): кандидатов=7544 -> корр=84 -> коинт=2 -> HL=2 -> Hurst=2
Итерация 6 (2016-06-01 -> 2017-05-31): кандидатов=7544 -> корр=97 -> коинт=4 -> HL=1 -> Hurst=1
Итерация 7 (2016-07-01 -> 2017-06-30): кандидатов=7544 -> корр=62 -> коинт=2 -> HL=1 -> Hurst=1
Итерация 8 (2016-08-01 -> 2017-07-31): кандидатов=7560 -> корр=60 -> коинт=1 -> HL=1 -> Hurst=1
Итерация 9 (2016-09-01 -> 2017-08-31): кандидатов=7576 -> корр=62 -> коинт=0 -> HL=0 -> Hurst=0
Итерация 10 (2016-10-01 -> 2017-09-30): кандидатов=7576 -> корр=63 -> коинт=0 -> HL=0 -> Hurst=0
Итерация 11 (2016-11-01 -> 2017-10-31

In [33]:
# Сохранение результата — один раз, после цикла по всем итерациям.
# Путь зависит от scenario_name (CSV/settings.csv), чтобы разные сценарии не перетирали друг друга.
# Всё пишется в один файл IS_{scenario_name}.xlsx с тремя листами:
#   Results               — сами отобранные пары
#   Funnel_Summary        — агрегированная воронка (в стиле Table 5 из диплома)
#   Funnel_Per_Iteration  — та же воронка по каждой из 108 итераций отдельно

import os
output_dir = f'DATA/Output/{scenario_name}'
os.makedirs(output_dir, exist_ok=True)

final_results = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()

funnel_df = pd.DataFrame(funnel_stats)
stages = [
    ('1. Correlation',   'N_Total_Pairs',         'N_After_Correlation'),
    ('2. Cointegration', 'N_After_Correlation',   'N_After_Cointegration'),
    ('3. Half-Life',     'N_After_Cointegration', 'N_After_HalfLife'),
    ('4. Hurst',         'N_After_HalfLife',      'N_After_Hurst'),
]
summary_rows = []
for name, col_in, col_out in stages:
    avg_in = funnel_df[col_in].mean()
    avg_out = funnel_df[col_out].mean()
    eliminated_pct = (1 - avg_out / avg_in) * 100 if avg_in > 0 else 0.0
    summary_rows.append({
        'Filter': name,
        'Avg_Input': round(avg_in, 1),
        'Avg_Output': round(avg_out, 1),
        'Eliminated_%': round(eliminated_pct, 1),
    })
funnel_summary_df = pd.DataFrame(summary_rows)

output_path = f'{output_dir}/IS_{scenario_name}.xlsx'
with pd.ExcelWriter(output_path) as writer:
    final_results.to_excel(writer, sheet_name='Results', index=False)
    funnel_summary_df.to_excel(writer, sheet_name='Funnel_Summary', index=False)
    funnel_df.to_excel(writer, sheet_name='Funnel_Per_Iteration', index=False)

print(f"[OK] Сохранено {len(final_results)} пар и статистика воронки в {output_path}")
print(funnel_summary_df.to_string(index=False))

[OK] Сохранено 1581 пар и статистика воронки в DATA/Output/ULTRA/IS_ULTRA.xlsx
          Filter  Avg_Input  Avg_Output  Eliminated_%
  1. Correlation    10003.6       592.2          94.1
2. Cointegration      592.2        28.3          95.2
    3. Half-Life       28.3        18.1          35.9
        4. Hurst       18.1        14.6          19.2
